In [3]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START, END
from typing import Literal,TypedDict
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.2)

In [5]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(..., description="The sentiment of the review")  

In [6]:
structured_model = model.with_structured_output(SentimentSchema)

In [7]:
prompt = "The food was great but the service was terrible. What is the sentiment of this review?"

structured_model.invoke(prompt).sentiment

'negative'

In [8]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis : dict
    response : str

In [9]:
def find_sentiment(state: ReviewState) -> ReviewState:
    prompt = f"What is the sentiment of this review? {state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {"sentiment": sentiment}

In [10]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)

graph.add_edge(START, "find_sentiment")
graph.add_edge("find_sentiment", END)

workflow = graph.compile()

In [11]:
initial_state = {
    'review':"The product was really good "
}

workflow.invoke(initial_state)

{'review': 'The product was really good ', 'sentiment': 'positive'}